# Module 11: Communicating results

**Goal:** make your analysis **understood, trusted and reproducible**. A brilliant analysis nobody can follow, check or rerun has little value.

**You will learn:** structuring a report · turning numbers into honest sentences · charts that make a point · docstrings and READMEs ·
tests and data validation for analysis code · reproducibility habits · ethics and privacy · giving a short talk

**Maths in this module:** none new. It's about *using* what you know carefully: rounding, intervals, effect sizes and uncertainty in plain words.

> ✍️ Rule of thumb: **if the reader must work to find your answer, you haven't finished the analysis.**

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import calendar
import doctest
import importlib.metadata as md
import math
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="colorblind")
students = pd.read_csv(DATA / "students.csv")
rides = pd.read_csv(DATA / "daily_rides.csv", parse_dates=["date"])

## 1. Know your reader, put the answer first

Ask: *Who is reading? What do they need to decide or learn? How much time do they have?* A manager wants the answer and the risk; a fellow analyst wants the method.
Use the **inverted pyramid**: the most important thing first, details later.

| section | one question it answers | typical length |
|---|---|---|
| **Summary** (top of the page) | What did you find and what should I do? | 3–5 sentences |
| **Question & data** | What were you asking, using what? | short paragraph + data card |
| **Methods** | How did you get the answer? | enough to reproduce |
| **Findings** | What did you see (charts + numbers with uncertainty)? | one subsection per finding |
| **Limitations** | Where might this be wrong? | honest and specific |
| **Next steps / conclusion** | So what now? | 2–4 bullets |

**Weak:** *"We performed an analysis of the relationship between several variables and exam scores using regression."*
**Strong:** *"Study time is the strongest predictor of exam scores: each extra weekly hour goes with about 1.6 more points (95% CI 1.4 to 1.7), after accounting for sleep, attendance and job status."*

The strong version has the **finding, the size, the uncertainty and the conditions**, in one sentence.

## 2. Turning numbers into honest sentences

Guidelines:

* **Round sensibly** (usually 2–3 significant digits). `2.9 points` beats `2.8997 points`. Don't show false precision.
* **Always give units** and say *what is being compared*.
* **Pair every estimate with its uncertainty** (an interval) and, where relevant, the sample size.
* Describe the **size** of an effect, not only whether p < 0.05.
* Use **hedged language that matches the evidence**: "suggests", "is compatible with", "inconclusive". Avoid "proves", "causes" (unless you ran an experiment).

A powerful habit: **generate sentences from your results with code**, so the text can never drift from the numbers if the data change.

In [ ]:
def describe_difference(label_a, label_b, mean_a, mean_b, ci_low, ci_high, p, unit):
    '''One honest sentence about a difference in means (a minus b) with its 95% CI and p-value.'''
    diff = mean_a - mean_b
    direction = "higher" if diff > 0 else "lower"
    p_text = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
    text = (f"On average, {label_a} had {abs(diff):.1f} {unit} {direction} than {label_b} "
            f"(95% CI {ci_low:.1f} to {ci_high:.1f}; {p_text}).")
    if ci_low <= 0 <= ci_high:
        text += " The data are compatible with no difference, so the result is inconclusive."
    return text

yes = students.loc[students["part_time_job"] == "yes", "exam_score"]
no = students.loc[students["part_time_job"] == "no", "exam_score"]
t = stats.ttest_ind(yes, no, equal_var=False)
low, high = t.confidence_interval(0.95)

print(describe_difference("students with a part-time job", "students without", yes.mean(), no.mean(),
                          low, high, t.pvalue, "points"))

Change the data and the sentence updates itself, including switching to "inconclusive" when the interval crosses zero.

## 3. Charts that make a point

A chart for **exploring** can be rough. A chart for **communicating** should carry **one message**. Compare:

In [ ]:
monthly = rides.groupby(rides["date"].dt.month)["rides"].mean()
peak_month = int(monthly.idxmax())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

# BEFORE: default styling, vague title, the reader must hunt for the point
axes[0].bar(monthly.index, monthly.values, color="tab:blue", edgecolor="black")
axes[0].set_title("Rides by month")

# AFTER: the title states the takeaway; the key bar is highlighted; labels, units; clutter removed
colors = ["#bbbbbb"] * 12
colors[peak_month - 1] = "#0072B2"
axes[1].bar(calendar.month_abbr[1:], monthly.values, color=colors)
axes[1].set_title(f"Ridership peaks in {calendar.month_name[peak_month]} at about {monthly.max():,.0f} rides a day, "
                  f"{monthly.max() / monthly.min():.1f}× the winter low", loc="left", fontsize=11)
axes[1].set_ylabel("Average rides per day")
axes[1].annotate("peak", (peak_month - 1, monthly.max()), textcoords="offset points", xytext=(0, 5), ha="center")
sns.despine(ax=axes[1])
axes[1].grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

**Checklist for a communication chart**

- **Title = the takeaway** (a sentence with the key number), not just a topic
- One message per chart; **highlight** what matters, grey out the rest
- Label axes with **units**; label lines/bars **directly** rather than using a legend where you can
- Bars start at zero; remove chart junk (extra gridlines, borders, 3-D)
- Use **colour-blind-safe** colours and don't rely on colour alone
- Add **alt text** or a caption when publishing (what a reader who can't see the chart needs to know)
- Export at a decent resolution: `fig.savefig("name.png", dpi=200, bbox_inches="tight")`

## 4. Docstrings, type hints and READMEs

A **docstring** is the first thing a future reader (you!) sees. Use a consistent style, such as Google style:

In [ ]:
def percent_change(old: float, new: float) -> float:
    '''Return the percent change from ``old`` to ``new``.

    Args:
        old: The starting value. Must not be zero.
        new: The ending value.

    Returns:
        (new - old) / old * 100, so a rise from 80 to 100 is 25.0.

    Raises:
        ValueError: If ``old`` is zero.

    Examples:
        >>> percent_change(80, 100)
        25.0
        >>> percent_change(100, 80)
        -20.0
    '''
    if old == 0:
        raise ValueError("old must not be zero")
    return (new - old) / old * 100

help(percent_change)

The **`Examples`** in a docstring can be executed as tests with Python's `doctest` module, so documentation and code can't silently disagree:

In [ ]:
finder, runner = doctest.DocTestFinder(), doctest.DocTestRunner()
for test in finder.find(percent_change, "percent_change", globs={"percent_change": percent_change}):
    runner.run(test)
results = runner.summarize(verbose=False)
assert results.failed == 0 and results.attempted == 2, results
print(results, "-> every example in the docstring is correct")

### The project README

Your README is the front door. A stranger should be able to answer in two minutes: *what is this, what did you find, and how do I run it?*
The template is in [`projects/templates/PROJECT_README.md`](../../projects/templates/PROJECT_README.md). It has these sections:

1. **Question** (one sentence) and a **summary of findings** (3–5 bullets with numbers)
2. **Data**: source, licence, how to get it (link to the data card)
3. **How to run**: exact commands, tested from a fresh clone
4. **Project structure**: a short tree
5. **Methods** in a few lines · **Results** with 1–3 key figures · **Limitations**
6. **Credits & licence**

## 5. Tests and data validation for analysis code

Two different safety nets:

| | **tests** (`pytest`) | **data validation** |
|---|---|---|
| check | your *code* on small, hand-made inputs | your *actual data* each time the pipeline runs |
| catches | wrong formulas, off-by-one, bad edge cases | unexpected values, schema changes, new typos in the data |
| when | before you trust a function | every run of `run_analysis.py` |

`pytest` features you'll use most: plain `assert`, `pytest.approx` for floats, `pytest.raises` for expected errors, **`parametrize`** to run one test on many inputs, and **fixtures** for shared setup.

In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="module11_"))
(workdir / "test_demo.py").write_text(
    "import pytest\n"
    "\n"
    "def percent_change(old, new):\n"
    "    if old == 0:\n"
    "        raise ValueError('old must not be zero')\n"
    "    return (new - old) / old * 100\n"
    "\n"
    "@pytest.mark.parametrize('old, new, expected', [\n"
    "    (100, 150, 50.0),\n"
    "    (200, 100, -50.0),\n"
    "    (80, 80, 0.0),\n"
    "])\n"
    "def test_percent_change(old, new, expected):\n"
    "    assert percent_change(old, new) == pytest.approx(expected)\n"
    "\n"
    "def test_zero_old_raises():\n"
    "    with pytest.raises(ValueError):\n"
    "        percent_change(0, 1)\n"
)
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"],
                        capture_output=True, text=True, cwd=workdir)
assert result.returncode == 0, result.stdout
print(result.stdout)

**What to test:** the functions where a silent error would change your conclusions: cleaning steps, anything with a formula, anything with boundaries.
Work out the expected answer **by hand first**, test edge cases (zero, empty, missing, exactly-on-the-boundary), and
**when you find a bug, write the failing test first.** (The hands-on exercise in `exercise/` measures how good your tests are by injecting bugs.)

### Data validation: assertions about the real data

In [ ]:
def validate_students(df: pd.DataFrame) -> list[str]:
    '''Return a list of problems found in the students table (empty list = all good).'''
    problems = []
    if df["student_id"].duplicated().any():
        problems.append("duplicate student_id values")
    if not df["exam_score"].between(0, 100).all():
        problems.append("exam_score outside 0-100")
    if not set(df["part_time_job"].unique()) <= {"yes", "no"}:
        problems.append("unexpected values in part_time_job")
    if df[["study_hours_week", "sleep_hours"]].isna().any().any():
        problems.append("missing study or sleep values")
    return problems

print("problems found:", validate_students(students))
broken = students.copy()
broken.loc[0, "exam_score"] = 140
print("after corrupting one row:", validate_students(broken))

Call a validator at the end of your cleaning step and **stop loudly** if it finds problems (`raise ValueError(...)`), rather than silently carrying bad data into your results.

## 6. Reproducibility: could a stranger get your numbers?

**Checklist**

- [ ] **One command** reproduces everything: `python run_analysis.py`
- [ ] **Random seeds** are set (`default_rng(seed)`, `random_state=...`)
- [ ] **Paths are relative** (no `C:\Users\me\...`); data location documented; raw data never edited
- [ ] **Dependencies listed** with versions (`requirements.txt`); Python version noted
- [ ] Notebooks run top-to-bottom after **Restart & Run All**
- [ ] **Tested from a fresh clone** in a new virtual environment
- [ ] Results are **generated by code**, not pasted by hand
- [ ] Data **provenance** recorded (source, date downloaded, licence)

Record your environment with a small function, and save it next to your results:

In [ ]:
def environment_report(packages=("numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "matplotlib")) -> dict:
    '''Python and package versions, for reproducibility notes.'''
    report = {"python": sys.version.split()[0]}
    for name in packages:
        try:
            report[name] = md.version(name)
        except md.PackageNotFoundError:
            report[name] = "not installed"
    return report

environment_report()

To turn a notebook into a shareable report (and prove it runs end to end), execute and export it:

```bash
jupyter nbconvert --to html --execute notebooks/02_report.ipynb
```

**Git hygiene:** commit small and often with messages that say *why* (`Fix share_missing for empty data`), keep large or generated files out via `.gitignore`, and never commit secrets or personal data.

## 7. Ethics, privacy and honest reporting

Data are about **people and the world**; analysis choices have consequences.

* **Privacy & consent:** do you have the right to use this data? Remove direct identifiers (names, emails, IDs). Beware **small groups**: a row like "the only 61-year-old in the 3rd-floor lab" identifies someone even without a name.
* **Representation & bias:** who is in your data, and who is missing? Do your conclusions apply beyond the sample? Could a model treat groups unfairly (different error rates)?
* **Honesty:** report what you *tried*, not just what worked (Module 8's multiple-comparisons trap). Don't cherry-pick the time window, the subgroup or the chart axis. Show uncertainty. Share limitations before critics find them.
* **Credit:** cite data sources, libraries and ideas you used. Respect licences.
* **Impact:** who could be helped or harmed if your result were acted upon? Prefer plain, cautious wording for high-stakes claims.

A simple privacy habit: **suppress small counts** before publishing a table.

In [ ]:
def suppress_small_groups(counts: pd.Series, min_n: int = 5) -> pd.Series:
    '''Replace group counts below ``min_n`` with NaN so small groups cannot be singled out.'''
    out = counts.astype(float).copy()
    out[out < min_n] = np.nan
    return out

counts = pd.Series({"Arts": 48, "Business": 45, "Engineering": 56, "Science": 51, "Astro": 3})
print(suppress_small_groups(counts))

The full checklist is in [`projects/templates/ETHICS_CHECKLIST.md`](../../projects/templates/ETHICS_CHECKLIST.md).

## 8. Presenting in five minutes

1. **Hook (30 s):** the question and why anyone should care
2. **Answer first (30 s):** your main finding in one sentence
3. **Evidence (2½ min):** 2–3 charts, each with a takeaway title and the key number with its interval
4. **Limits (45 s):** what might be wrong and what you'd do next
5. **Ask / next step (30 s):** what should the audience do or think now?

Practice aloud, once. Cut anything the audience doesn't need. A demo (`python run_analysis.py`) that runs live earns a lot of trust. Outline: [`projects/templates/PRESENTATION.md`](../../projects/templates/PRESENTATION.md).

## Recap

* Lead with the answer; one message per chart; numbers always with units and uncertainty; hedge in proportion to the evidence.
* Docstrings + README + data card make your work usable; tests and validation make it trustworthy; seeds, relative paths and one command make it reproducible.
* Report honestly, protect people, and credit your sources.

**Next:** `exercises.ipynb`, the hands-on [`exercise/`](exercise/README.md) (write tests that catch bugs), then the **project checkpoint** in the README.